## 1. Imports & Setup


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')


## 2. Load Dataset


In [8]:
DATA_PATH = '../global_bleaching_environmental.csv'

data = pd.read_csv(DATA_PATH, na_values=['nd', 'NA', ''])
print(f"Raw shape: {data.shape}")
print(f"\nFirst few rows:")

Raw shape: (41361, 62)

First few rows:


In [9]:
data.shape

(41361, 62)

In [10]:
drop_columns = ['Site_ID', 'Sample_ID', 'Data_Source','Ocean_Name', 'Reef_ID', 'Realm_Name',
       'Ecoregion_Name', 'Country_Name', 'State_Island_Province_Name',
       'City_Town_Name', 'Site_Name', 'Date_Day',
       'Substrate_Name', 'Site_Comments', 'Sample_Comments',
       'Bleaching_Comments', 'Bleaching_Level', 'Percent_Cover', 'SSTA_Mean', 'Exposure',
       ] # Exposure - interesting

data.drop(columns= drop_columns, inplace= True)


In [11]:
data = data[data['SSTA'] > 0]

data = data[data['Date_Year'] >= 2003]

PAPER_FEATURES = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']
TARGET = 'Percent_Bleaching'
data = data.dropna(subset=['Percent_Bleaching'])


In [12]:
data.shape

(19727, 42)

In [13]:
print(data[TARGET].isna().any().any())
print(data[PAPER_FEATURES].isna().any().any())

False
True


## 6. Train/Test Split


In [14]:
from sklearn.impute import SimpleImputer

# Train/test split ก่อน
X = data[PAPER_FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Impute (fit เฉพาะ train, transform ทั้งคู่)
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)

# Optuna

In [15]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, VotingRegressor

In [16]:
def rf_objective(trial):

    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 5, 40),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

rf_study = optuna.create_study(direction="maximize")

rf_study.optimize(rf_objective, n_trials=50)

print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

[I 2026-05-06 10:17:39,376] A new study created in memory with name: no-name-d202d173-af74-4f28-9aab-229f08ac098c
[I 2026-05-06 10:17:42,858] Trial 0 finished with value: 0.5626057386171353 and parameters: {'n_estimators': 255, 'max_depth': 36, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5626057386171353.
[I 2026-05-06 10:17:45,675] Trial 1 finished with value: 0.5125700059823366 and parameters: {'n_estimators': 258, 'max_depth': 12, 'min_samples_split': 3, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5626057386171353.
[I 2026-05-06 10:17:48,964] Trial 2 finished with value: 0.5584822508570636 and parameters: {'n_estimators': 277, 'max_depth': 29, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.5626057386171353.
[I 2026-05-06 10:17:49,245] Trial 3 finished with value: 0.33348159159784185 and parameters: {'n_estimators': 57, 'max_depth': 6, 'mi

========== RF BEST ==========
{'n_estimators': 182, 'max_depth': 27, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Best R²: 0.6134


## 8. USE Baseline

In [20]:
rf_model = RandomForestRegressor(
    n_estimators=182,
    max_depth=27,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1,
    random_state=42
)
# Paste rf_study.best_params here as comment

rf_model.fit(X_train, y_train)
print("Model trained.")

Model trained.


## 9. Evaluate Model

In [21]:
y_pred = rf_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("=" * 50)
print("BASELINE RESULTS (single seed = 42)")
print("=" * 50)
print(f"R²    = {r2:.4f}    (paper: 0.25)")
print(f"RMSE  = {rmse:.4f}    (paper: 7.91)")
print(f"MAE   = {mae:.4f}")
print(f"Accuracy (100 - MAE) = {100 - mae:.2f}%   (paper: 96%)")
print(f"\n⚠️  Single-seed result อาจแกว่ง — ดู multi-seed check ด้านล่าง")

BASELINE RESULTS (single seed = 42)
R²    = 0.6743    (paper: 0.25)
RMSE  = 10.8471    (paper: 7.91)
MAE   = 5.1803
Accuracy (100 - MAE) = 94.82%   (paper: 96%)

⚠️  Single-seed result อาจแกว่ง — ดู multi-seed check ด้านล่าง


## 10. K-Fold Cross Validation (k=4)


In [24]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

cv = KFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

# ✅ Wrap ใน Pipeline เพื่อให้ imputer fit ใหม่ทุก fold
rf_cv = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(
    n_estimators=182,
    max_depth=27,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1,
    random_state=42
))
])

cv_results = cross_validate(
    rf_cv,
    X,
    y,
    cv=cv,
    scoring={
        'r2': 'r2',
        'neg_mse': 'neg_mean_squared_error',
        'neg_mae': 'neg_mean_absolute_error'
    },
    n_jobs=-1,
    return_train_score=False
)


kfold_df = pd.DataFrame({
    'fold': range(1, 5),
    'R²': cv_results['test_r2'],
    'RMSE': np.sqrt(-cv_results['test_neg_mse']),
    'MAE': -cv_results['test_neg_mae']
})
kfold_df['Accuracy'] = 100 - kfold_df['MAE']

print("K-Fold Cross Validation (k=4):")
print(kfold_df.to_string(index=False))
print()
print(f"Mean R²:       {kfold_df['R²'].mean():.4f} ± {kfold_df['R²'].std():.4f}")
print(f"Mean RMSE:     {kfold_df['RMSE'].mean():.4f} ± {kfold_df['RMSE'].std():.4f}")
print(f"Mean MAE:      {kfold_df['MAE'].mean():.4f} ± {kfold_df['MAE'].std():.4f}")
print(f"Mean Accuracy: {kfold_df['Accuracy'].mean():.2f}% ± {kfold_df['Accuracy'].std():.2f}%")

K-Fold Cross Validation (k=4):
 fold       R²      RMSE      MAE  Accuracy
    1 0.653439 11.459539 5.488310 94.511690
    2 0.644487 11.729000 5.590378 94.409622
    3 0.660234 11.480205 5.463088 94.536912
    4 0.648606 11.785628 5.633026 94.366974

Mean R²:       0.6517 ± 0.0068
Mean RMSE:     11.6136 ± 0.1678
Mean MAE:      5.5437 ± 0.0811
Mean Accuracy: 94.46% ± 0.08%
